In [117]:
%load_ext autoreload
%autoreload 2


import numpy as np
from astropy.table import Table
from astropy import units as u
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display, HTML
from crossmatching import Crossmatcher, EMCCatalog, EMCIdSupplier, NEACatalog, SimbadIdSupplier, ParamFiller, temperate_mask, rocky_mask
from crossmatching.enrichment import (
    HpicParamSource, NeaParamSource, SimbadParamSource,
    EpicParamSource, ToiParamSource, EuParamSource
)
from collections import Counter
import glob
import ipydatagrid
import matplotlib

mpl.rcParams['figure.dpi'] = 300

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [118]:
# Habitable zone flux limits in S/S_earth — Kopparapu et al. (2014), optimistic bounds
HZ_INNER = 1.7   # inner (hot) edge — Recent Venus
HZ_OUTER = 0.35  # outer (cold) edge — Early Mars

# Set True to include planets where the flux_rel ± uncertainty interval overlaps
# the HZ (i.e. could be temperate within measurement error), False for strict central-value test.
USE_INTERVAL_HZ = True

# Set True to include planets with only msini-derived radius bounds overlapping
# the rocky range (uncertain rocky), False for confirmed transit radius only.
USE_INTERVAL_ROCKY = True

In [119]:
input_table = Table.read('./input/HPIC_LC4_combined_d50.txt', format='ascii',)
# cme = Crossmatcher(EMCCatalog(), SimbadIdSupplier(),
#     coordinate_search_radius=50*u.arcsec
# )
cme = Crossmatcher(EMCCatalog(), EMCIdSupplier())
cme.load_catalog(from_file='./exo-mercat.csv', format='csv')
# _ = cme.load_alternate_ids(input_table['star_name'].tolist(), from_file='./alternate_ids_hpic.txt')
cme.load_alternate_ids(input_table['star_name'].tolist(), from_file='exo-mercat.csv')
input_table = cme.remove_duplicates(input_table, input_starname_key='star_name')
out_emc = cme.combined_crossmatch(input_table, input_starname_key='star_name')

cm = Crossmatcher(NEACatalog(), SimbadIdSupplier())
cm.load_catalog(from_file="./pscomppars.txt")

objectid,pl_name,pl_letter,hostid,hostname,hd_name,hip_name,tic_id,disc_pubdate,disc_year,disc_method,discoverymethod,disc_locale,disc_facility,disc_instrument,disc_telescope,disc_refname,ra,raerr1,raerr2,rasymerr,rastr,ra_solnid,ra_reflink,dec,decerr1,decerr2,decsymerr,decstr,dec_solnid,dec_reflink,glon,glonerr1,glonerr2,glonsymerr,glonstr,glon_solnid,glon_reflink,glat,glaterr1,glaterr2,glatsymerr,glatstr,glat_solnid,glat_reflink,elon,elonerr1,elonerr2,elonsymerr,elonstr,elon_solnid,elon_reflink,elat,elaterr1,elaterr2,elatsymerr,elat_solnid,elat_reflink,elatstr,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbpersymerr,pl_orbperlim,pl_orbperstr,pl_orbperformat,pl_orbper_solnid,pl_orbper_reflink,pl_orblpererr1,pl_orblper,pl_orblpererr2,pl_orblpersymerr,pl_orblperlim,pl_orblperstr,pl_orblperformat,pl_orblper_solnid,pl_orblper_reflink,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxsymerr,pl_orbsmaxlim,pl_orbsmaxstr,pl_orbsmaxformat,pl_orbsmax_solnid,pl_orbsmax_reflink,pl_orbincl,pl_orbinclerr1,pl_orbinclerr2,pl_orbinclsymerr,pl_orbincllim,pl_orbinclstr,pl_orbinclformat,pl_orbincl_solnid,pl_orbincl_reflink,pl_orbtper,pl_orbtpererr1,pl_orbtpererr2,pl_orbtpersymerr,pl_orbtperlim,pl_orbtperstr,pl_orbtperformat,pl_orbtper_solnid,pl_orbtper_reflink,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccensymerr,pl_orbeccenlim,pl_orbeccenstr,pl_orbeccenformat,pl_orbeccen_solnid,pl_orbeccen_reflink,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtsymerr,pl_eqtlim,pl_eqtstr,pl_eqtformat,pl_eqt_solnid,pl_eqt_reflink,pl_occdep,pl_occdeperr1,pl_occdeperr2,pl_occdepsymerr,pl_occdeplim,pl_occdepstr,pl_occdepformat,pl_occdep_solnid,pl_occdep_reflink,pl_insol,pl_insolerr1,pl_insolerr2,pl_insolsymerr,pl_insollim,pl_insolstr,pl_insolformat,pl_insol_solnid,pl_insol_reflink,pl_dens,pl_denserr1,sy_umagerr1,sy_umagerr2,sy_umaglim,sy_umagsymerr,sy_umagstr,sy_umagformat,sy_umag_solnid,sy_umag_reflink,sy_rmag,sy_rmagerr1,sy_rmagerr2,sy_rmaglim,sy_rmagsymerr,sy_rmagstr,sy_rmagformat,sy_rmag_solnid,sy_rmag_reflink,sy_imag,sy_imagerr1,sy_imagerr2,sy_imaglim,sy_imagsymerr,sy_imagstr,sy_imagformat,sy_imag_solnid,sy_imag_reflink,sy_zmag,sy_zmagerr1,sy_zmagerr2,sy_zmaglim,sy_zmagsymerr,sy_zmagstr,sy_zmagformat,sy_zmag_solnid,sy_zmag_reflink,sy_w1mag,sy_w1magerr1,sy_w1magerr2,sy_w1maglim,sy_w1magsymerr,sy_w1magstr,sy_w1magformat,sy_w1mag_solnid,sy_w1mag_reflink,sy_w2mag,sy_w2magerr1,sy_w2magerr2,sy_w2maglim,sy_w2magsymerr,sy_w2magstr,sy_w2magformat,sy_w2mag_solnid,sy_w2mag_reflink,sy_w3mag,sy_w3magerr1,sy_w3magerr2,sy_w3maglim,sy_w3magsymerr,sy_w3magstr,sy_w3magformat,sy_w3mag_solnid,sy_w3mag_reflink,sy_w4mag,sy_w4magerr1,sy_w4magerr2,sy_w4maglim,sy_w4magsymerr,sy_w4magstr,sy_w4magformat,sy_w4mag_solnid,sy_w4mag_reflink,sy_gmag,sy_gmagerr1,sy_gmagerr2,sy_gmaglim,sy_gmagsymerr,sy_gmagstr,sy_gmagformat,sy_gmag_solnid,sy_gmag_reflink,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,sy_gaiamaglim,sy_gaiamagsymerr,sy_gaiamagstr,sy_gaiamagformat,sy_gaiamag_solnid,sy_gaiamag_reflink,sy_tmag,sy_tmagerr1,sy_tmagerr2,sy_tmaglim,sy_tmagsymerr,sy_tmagstr,sy_tmagformat,sy_tmag_solnid,sy_tmag_reflink,sy_name,pl_controv_flag,pl_orbtper_systemref,pl_tranmid_systemref,st_metratio,st_spectype,st_spectype_solnid,st_spectype_reflink,sy_plxlim,sy_kepmag,sy_kepmagerr1,sy_kepmagerr2,sy_kepmaglim,sy_kepmagsymerr,sy_kepmagstr,sy_kepformat,sy_kepmag_solnid,sy_kepmag_reflink,st_rotp,st_rotperr1,st_rotperr2,st_rotpsymerr,st_rotplim,st_rotpstr,st_rotpformat,st_rotp_solnid,st_rotp_reflink,pl_projobliq,pl_projobliqerr1,pl_projobliqerr2,pl_projobliqsymerr,pl_projobliqlim,pl_projobliqstr,pl_projobliqformat,pl_denserr2,pl_denssymerr,pl_denslim,pl_densstr,pl_densformat,pl_dens_solnid,pl_dens_reflink,pl_trandep,pl_trandeperr1,pl_trandeperr2,pl_trandepsymerr,pl_trandeplim,pl_trandepstr,pl_trandepformat,pl_trandep_solnid,pl_trandep_reflink,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_tranmidsymerr,pl_tranmidlim,pl_tranmidstr,pl_tranmidformat,pl_tranmid_solnid,pl_tranmid_reflink,pl_trandur,pl_trandurerr1,pl_trandurerr2,pl_tr

In [120]:
# ── Tier 1: NEA (pscomppars) ──────────────────────────────────────────────────
nea_src = NeaParamSource()
nea_src.load(from_file='./pscomppars.txt', format='ascii')
print(f'NEA planets loaded:     {len(nea_src._lookup):,}')
print(f'With insol:             {sum(1 for v in nea_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in nea_src._lookup.values() if "teff" in v and "rad" in v):,}')


# Tier 3
eu_src = EuParamSource()
eu_path = sorted(glob.glob('../Exo-MerCat/InputSources/eu_init*.csv'))[-1]
eu_src.load(from_file=eu_path, format="ascii.csv")
print(f"\nEU planets loaded: {len(eu_src._lookup):,}")

# ── Tier 2: EPIC (K2 planets not in NEA) ─────────────────────────────────────
epic_src = EpicParamSource()
epic_path = sorted(glob.glob('../Exo-MerCat/InputSources/epic_init*.csv'))[-1]
epic_src.load(from_file=epic_path, format='ascii.csv')
print(f'\nEPIC planets loaded:    {len(epic_src._lookup):,}')
print(f'With insol:             {sum(1 for v in epic_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in epic_src._lookup.values() if "teff" in v and "rad" in v):,}')

# ── Tier 3: TOI (TESS objects) ────────────────────────────────────────────────
toi_src = ToiParamSource()
toi_path = sorted(glob.glob('../Exo-MerCat/InputSources/toi_init*.csv'))[-1]
toi_src.load(from_file=toi_path, format='ascii.csv')
print(f'\nTOI entries loaded:     {len(toi_src._lookup):,}')
print(f'With insol:             {sum(1 for v in toi_src._lookup.values() if "insol" in v):,}')
print(f'With teff + rad:        {sum(1 for v in toi_src._lookup.values() if "teff" in v and "rad" in v):,}')

# ── Tier 4: SIMBAD fallback ───────────────────────────────────────────────────
simbad_src = SimbadParamSource()
simbad_src.load(from_file='simbad_params.txt')
print(f'SIMBAD matches: {len(simbad_src._lookup):,}')

full_emc   = cme.catalog_table
nasa_names = [str(n).strip() for n in full_emc['nasa_name']]
nea_keys   = set(nea_src._lookup)
has_nea    = np.array([n in nea_keys for n in nasa_names])

emc_only_main_ids = sorted(set(
    str(full_emc['main_id'][i]).strip()
    for i in np.where(~has_nea)[0]
    if str(full_emc['main_id'][i]).strip()
))

print(f'\nFull EMC: {len(full_emc):,} planets')
print(f'Matched to NEA (nasa_name): {has_nea.sum():,} ({100*has_nea.mean():.1f}%)')
print(f'EMC-only unique host main_ids: {len(emc_only_main_ids):,}')



hpic_src = HpicParamSource(out_emc)
hpic_src.load()

merger   = ParamFiller([nea_src, eu_src, epic_src, toi_src, simbad_src, hpic_src])
e_emc  = merger.enrich(
    cme.catalog_table,
    input_starname_key="star_name", 
    id_supplier=cme.id_supplier, 
    alternate_ids=cme.alternate_ids,
    **EMCCatalog.ENRICH_KEYS
)



NEA planets loaded:     6,298
With insol:             4,413
With teff + rad:        5,977

EU planets loaded: 11,159

EPIC planets loaded:    1,806
With insol:             175
With teff + rad:        1,230

TOI entries loaded:     7,931
With insol:             7,752
With teff + rad:        7,394
SIMBAD matches: 8,930

Full EMC: 16,097 planets
Matched to NEA (nasa_name): 6,219 (38.6%)
EMC-only unique host main_ids: 9,223


In [121]:
e_nea = merger.enrich(
    cm.catalog_table,
    input_starname_key="pl_name",
    id_supplier=cm.id_supplier,
    **NEACatalog.ENRICH_KEYS
)

In [122]:
((e_emc["st_mass_src"] == '') * (e_emc["st_lum_src"] != '')).sum()

np.int64(901)

In [123]:
e_nea.colnames

['objectid',
 'pl_name',
 'pl_letter',
 'hostid',
 'hostname',
 'hd_name',
 'hip_name',
 'tic_id',
 'disc_pubdate',
 'disc_year',
 'disc_method',
 'discoverymethod',
 'disc_locale',
 'disc_facility',
 'disc_instrument',
 'disc_telescope',
 'disc_refname',
 'ra',
 'raerr1',
 'raerr2',
 'rasymerr',
 'rastr',
 'ra_solnid',
 'ra_reflink',
 'dec',
 'decerr1',
 'decerr2',
 'decsymerr',
 'decstr',
 'dec_solnid',
 'dec_reflink',
 'glon',
 'glonerr1',
 'glonerr2',
 'glonsymerr',
 'glonstr',
 'glon_solnid',
 'glon_reflink',
 'glat',
 'glaterr1',
 'glaterr2',
 'glatsymerr',
 'glatstr',
 'glat_solnid',
 'glat_reflink',
 'elon',
 'elonerr1',
 'elonerr2',
 'elonsymerr',
 'elonstr',
 'elon_solnid',
 'elon_reflink',
 'elat',
 'elaterr1',
 'elaterr2',
 'elatsymerr',
 'elat_solnid',
 'elat_reflink',
 'elatstr',
 'pl_orbper',
 'pl_orbpererr1',
 'pl_orbpererr2',
 'pl_orbpersymerr',
 'pl_orbperlim',
 'pl_orbperstr',
 'pl_orbperformat',
 'pl_orbper_solnid',
 'pl_orbper_reflink',
 'pl_orblpererr1',
 'pl_orbl

In [136]:
new_cols = list(set(e_nea.colnames) - set(cm.catalog_table.colnames))
src_cols = [i for i in e_nea.colnames if i.endswith("_src")]
for col in src_cols:
    print(col, ((e_nea[col] != 'input') & (e_nea[col] != '')).sum())
    print(Counter(e_nea[col]))

st_rad_src 60
Counter({np.str_('input'): 5980, np.str_(''): 258, np.str_('ms(teff:spectype_derived(spec:input))'): 16, np.str_('mann_teff(teff:spectype_derived(spec:input))'): 14, np.str_('ms(teff:input)'): 13, np.str_('logg_derived(mass:input logg:input)'): 9, np.str_('mann_mks(kmag:input)'): 5, np.str_('mann_teff(teff:input)'): 3})
st_mass_src 4
Counter({np.str_('input'): 6289, np.str_(''): 5, np.str_('logg_derived(rad:input logg:input)'): 4})
st_teff_src 37
Counter({np.str_('input'): 6004, np.str_(''): 257, np.str_('spectype_derived(spec:input)'): 37})
st_logg_src 0
Counter({np.str_('input'): 5976, np.str_(''): 322})
st_met_src 0
Counter({np.str_('input'): 5659, np.str_(''): 639})
st_lum_src 53
Counter({np.str_('input'): 5986, np.str_(''): 259, np.str_('derived(rad:ms(teff:spectype_derived(spec:input)) teff:spectype_derived(spec:input))'): 16, np.str_('derived(rad:mann_teff(teff:spectype_derived(spec:input)) teff:spectype_derived(spec:input))'): 14, np.str_('derived(rad:ms(teff:inpu

In [125]:
from collections import Counter
for k in Counter(e_emc["pl_insol_src"]).keys():
    k = str(k)
    print(sum([1 for i in k if i == "("]), k)

Counter(e_emc["pl_insol_src"])

2 derived(r:torres(teff:simbad logg:simbad) teff:simbad a:input)
0 toi
1 derived(lum:nea a:input)
1 derived(r:hpic teff:simbad a:input)
3 derived(r:hpic teff:simbad a:kepler(mass:logg_derived(rad:hpic logg:simbad) period:input))
1 derived(r:hpic teff:hpic a:input)
2 derived(r:mann_teff(teff:simbad) teff:simbad a:input)
4 derived(r:ms(teff:spectype_derived(spec:nea)) teff:spectype_derived(spec:nea) a:input)
5 derived(r:torres(teff:simbad logg:simbad) teff:simbad a:kepler(mass:logg_derived(rad:torres(teff:simbad logg:simbad) logg:simbad) period:input))
2 derived(r:ms(teff:simbad) teff:simbad a:input)
0 nea
0 
0 epic
4 derived(r:ms(teff:spectype_derived(spec:simbad)) teff:spectype_derived(spec:simbad) a:input)
3 derived(r:epic teff:epic a:kepler(mass:logg_derived(rad:epic logg:epic) period:input))
5 derived(r:mann_mks(kmag:simbad) teff:simbad a:kepler(mass:logg_derived(rad:mann_mks(kmag:simbad) logg:simbad) period:input))
3 derived(r:epic teff:simbad a:kepler(mass:logg_derived(rad:epic lo

Counter({np.str_('toi'): 5355,
         np.str_('nea'): 4405,
         np.str_(''): 1809,
         np.str_('derived(r:torres(teff:simbad logg:simbad) teff:simbad a:input)'): 1440,
         np.str_('derived(lum:nea a:input)'): 1366,
         np.str_('derived(r:epic teff:simbad a:kepler(mass:logg_derived(rad:epic logg:simbad) period:input))'): 259,
         np.str_('derived(r:hpic teff:simbad a:input)'): 243,
         np.str_('derived(r:torres(teff:simbad logg:simbad) teff:simbad a:kepler(mass:logg_derived(rad:torres(teff:simbad logg:simbad) logg:simbad) period:input))'): 185,
         np.str_('derived(lum:nea a:kepler(mass:nea period:input))'): 153,
         np.str_('derived(r:epic teff:epic a:kepler(mass:epic period:input))'): 140,
         np.str_('derived(r:epic teff:epic a:kepler(mass:logg_derived(rad:epic logg:epic) period:input))'): 118,
         np.str_('derived(lum:epic a:kepler(mass:epic period:input))'): 106,
         np.str_('derived(r:ms(teff:simbad) teff:simbad a:input)'): 

In [126]:
e_emc["exo-mercat_name", "status", "nasa_name", "toi_name", "epic_name"][e_emc["a_src"] == "kepler(mass:logg_derived(rad:torres(teff:epic logg:simbad) logg:simbad) p:p)"]

exo-mercat_name,status,nasa_name,toi_name,epic_name
str39,str14,str28,str11,str19


In [127]:
e_emc["exo-mercat_name", "main_id_aliases", "status"][e_emc["pl_insol_src"] == "r:mann_mks(kmag:simbad) teff:simbad a:kepler(mass:logg_derived(rad:mann_mks(kmag:simbad) logg:simbad) p:p)"]

exo-mercat_name,main_id_aliases,status
str39,str1562,str14


In [128]:
sample_row = cme.catalog_table[cme.catalog_table["exo-mercat_name"] == "2MASS J15104786-2818174 AB b"]
sample_row

exo-mercat_name,nasa_name,toi_name,epic_name,eu_name,oec_name,host,letter,main_id,binary,main_id_ra,main_id_dec,mass,mass_max,mass_min,mass_url,msini,msini_max,msini_min,msini_url,bestmass,bestmass_max,bestmass_min,bestmass_url,bestmass_provenance,p,p_max,p_min,p_url,r,r_max,r_min,r_url,a,a_max,a_min,a_url,e,e_max,e_min,e_url,i,i_max,i_min,i_url,discovery_method,status,checked_status_string,original_status_string,confirmed,discovery_year,main_id_aliases,main_id_provenance,angular_separation_flag,angular_separation,catalog,duplicate_catalog_flag,duplicate_names,binary_coordinate_mismatch_flag,binary_complex_system_flag,coordinate_mismatch_flag,coordinate_mismatch,period_mismatch_flag,fallback_merge_flag,misnamed_duplicates_flag,row_update
str39,str28,str11,str19,str31,str27,str29,str3,str36,str6,float64,float64,float64,float64,float64,str50,float64,float64,float64,str21,float64,float64,float64,str50,str5,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,str39,str14,str95,str95,int64,int64,str1562,str11,int64,str49,str20,int64,str101,int64,int64,int64,int64,int64,int64,int64,str10
2MASS J15104786-2818174 AB b,--,--,--,2M1510 (AB)b,--,2MASS J1510,b,2MASS J15104786-2818174,AB,227.69940541649,-28.30486663737,0.031,--,--,eu,--,--,--,--,0.031,--,--,eu,Mass,100.0,--,--,eu,--,--,--,--,--,--,--,--,--,--,--,--,90.0,--,--,eu,Radial Velocity,CONFIRMED,eu: CONFIRMED,eu: CONFIRMED,1,2025,"2MASS J15104786-2818174,2MASSW J1510478-281817,2MUCD 20604,DENIS J151047.8-281817,Gaia DR2 6212595980928732032,Gaia DR3 6212595980928732032,TIC 61253912,WISE J151047.77-281817.9,WISEA J151047.76-281818.0",SIMBAD,0,eu: 0.0,eu,0,--,0,0,0,--,0,0,0,2026-05-31


In [129]:
simple_merger = ParamFiller([nea_src])
simple_merger.enrich(
    sample_row, 
    input_starname_key="star_name", 
    id_supplier=cme.id_supplier, 
    alternate_ids=cme.alternate_ids,
    **EMCCatalog.ENRICH_KEYS
)

exo-mercat_name,nasa_name,toi_name,epic_name,eu_name,oec_name,host,letter,main_id,binary,main_id_ra,main_id_dec,mass,mass_max,mass_min,mass_url,msini,msini_max,msini_min,msini_url,bestmass,bestmass_max,bestmass_min,bestmass_url,bestmass_provenance,p,p_max,p_min,p_url,r,r_max,r_min,r_url,a,a_max,a_min,a_url,e,e_max,e_min,e_url,i,i_max,i_min,i_url,discovery_method,status,checked_status_string,original_status_string,confirmed,discovery_year,main_id_aliases,main_id_provenance,angular_separation_flag,angular_separation,catalog,duplicate_catalog_flag,duplicate_names,binary_coordinate_mismatch_flag,binary_complex_system_flag,coordinate_mismatch_flag,coordinate_mismatch,period_mismatch_flag,fallback_merge_flag,misnamed_duplicates_flag,row_update,st_rad,st_rad_src,st_rad_err1,st_rad_err2,st_mass,st_mass_src,st_mass_err1,st_mass_err2,st_teff,st_teff_src,st_teff_err1,st_teff_err2,st_logg,st_logg_src,st_logg_err1,st_logg_err2,st_met,st_met_src,st_met_err1,st_met_err2,st_lum,st_lum_src,st_lum_err1,st_lum_err2,sy_vmag,sy_vmag_src,sy_vmag_err1,sy_vmag_err2,sy_kmag,sy_kmag_src,sy_kmag_err1,sy_kmag_err2,sy_dist,sy_dist_src,sy_dist_err1,sy_dist_err2,pl_insol,pl_insol_src,pl_insol_err1,pl_insol_err2,pl_eqt,pl_eqt_src,pl_eqt_err1,pl_eqt_err2,a_src,a_err1,a_err2,st_spectype,st_spectype_src,r_lower_bound,r_upper_bound,spectral_category
str39,str28,str11,str19,str31,str27,str29,str3,str36,str6,float64,float64,float64,float64,float64,str50,float64,float64,float64,str21,float64,float64,float64,str50,str5,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,float64,float64,float64,str67,str39,str14,str95,str95,int64,int64,str1562,str11,int64,str49,str20,int64,str101,int64,int64,int64,int64,int64,int64,int64,str10,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,float64,str1,float64,float64,str1,float64,float64,str1,str1,float64,float64,str5
2MASS J15104786-2818174 AB b,--,--,--,2M1510 (AB)b,--,2MASS J1510,b,2MASS J15104786-2818174,AB,227.69940541649,-28.30486663737,0.031,--,--,eu,--,--,--,--,0.031,--,--,eu,Mass,100.0,--,--,eu,--,--,--,--,--,--,--,--,--,--,--,--,90.0,--,--,eu,Radial Velocity,CONFIRMED,eu: CONFIRMED,eu: CONFIRMED,1,2025,"2MASS J15104786-2818174,2MASSW J1510478-281817,2MUCD 20604,DENIS J151047.8-281817,Gaia DR2 6212595980928732032,Gaia DR3 6212595980928732032,TIC 61253912,WISE J151047.77-281817.9,WISEA J151047.76-281818.0",SIMBAD,0,eu: 0.0,eu,0,--,0,0,0,--,0,0,0,2026-05-31,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,--,,--,--,,--,--,--,,--,--,Other


In [130]:
cm = Crossmatcher(NEACatalog(), SimbadIdSupplier())
cm.load_catalog(from_file="pscomppars.txt")
cm.load_alternate_ids(input_table["star_name"], from_file="alternate_ids_hpic.txt")

out_nea = cm.combined_crossmatch(input_table, input_starname_key="star_name")

In [131]:
e_nea = merger.enrich(
    cm.catalog_table,
    input_starname_key="star_name", 
    id_supplier=cm.id_supplier,
    alternate_ids=cm.alternate_ids,
    planet_radius_key='pl_radj',
    semi_major_axis_key='pl_orbsmax',
    period_key='pl_orbper'
)

new_cols = set(e_nea.colnames) - set(out_nea.colnames)
print(new_cols)
e_nea["pl_name", *new_cols]

{'pl_eqt_err2', 'st_met_src', 'st_mass_src', 'sy_vmag_err1', 'st_spectype_src', 'pl_insol_err1', 'st_rad_err2', 'st_rad_src', 'pl_eqt_err1', 'pl_insol_err2', 'sy_dist_err1', 'st_met_err2', 'pl_orbsmax_err1', 'pl_radj_upper_bound', 'st_lum_src', 'st_teff_err2', 'pl_eqt_src', 'st_teff_err1', 'sy_kmag_err2', 'st_lum_err2', 'st_logg_err1', 'sy_dist_src', 'sy_vmag_src', 'sy_kmag_err1', 'pl_radj_lower_bound', 'sy_kmag_src', 'st_teff_src', 'sy_vmag_err2', 'st_lum_err1', 'st_met_err1', 'spectral_category', 'st_mass_err1', 'pl_orbsmax_src', 'pl_insol_src', 'st_logg_err2', 'pl_orbsmax_err2', 'st_rad_err1', 'st_logg_src', 'sy_dist_err2', 'st_mass_err2'}


pl_name,pl_eqt_err2,st_met_src,st_mass_src,sy_vmag_err1,st_spectype_src,pl_insol_err1,st_rad_err2,st_rad_src,pl_eqt_err1,pl_insol_err2,sy_dist_err1,st_met_err2,pl_orbsmax_err1,pl_radj_upper_bound,st_lum_src,st_teff_err2,pl_eqt_src,st_teff_err1,sy_kmag_err2,st_lum_err2,st_logg_err1,sy_dist_src,sy_vmag_src,sy_kmag_err1,pl_radj_lower_bound,sy_kmag_src,st_teff_src,sy_vmag_err2,st_lum_err1,st_met_err1,spectral_category,st_mass_err1,pl_orbsmax_src,pl_insol_src,st_logg_err2,pl_orbsmax_err2,st_rad_err1,st_logg_src,sy_dist_err2,st_mass_err2
str29,float64,str5,str34,float64,str5,float64,float64,str44,float64,float64,float64,float64,float64,float64,str91,float64,str112,float64,float64,float64,float64,str5,str5,float64,float64,str5,str28,float64,float64,float64,str19,float64,str60,str97,float64,float64,float64,str5,float64,float64
Kepler-1167 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-1740 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-1581 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-644 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-1752 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-280 c,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-1208 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-263 c,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--
Kepler-1101 b,--,input,input,--,,--,--,input,--,--,--,--,--,--,input,--,input,--,--,--,--,input,input,--,--,input,input,--,--,--,Other,--,input,input,--,--,--,input,--,--


In [132]:
e_emc["st_spectype"][e_emc["nasa_name"] == "LkCa 15 b"]

K5:Ve


In [133]:
Counter(e_emc["st_spectype_src"])

Counter({np.str_(''): 10721,
         np.str_('simbad'): 2905,
         np.str_('nea'): 2282,
         np.str_('epic'): 141,
         np.str_('hpic'): 48})

In [134]:
for i in e_emc[merger.PARAM_NAMES_QUANTITIES].iterrows():
    for j in i:
        if j == "eu":
            print("hurray")

KeyError: 'pl_a'

In [ ]:
from crossmatching.enrichment.spectral_types import get_spectral_class_range, _TEFF_SPECTYPE, _parse_spt, standardize_spectral_type

for spectype, _  in _TEFF_SPECTYPE:
    print(f"{spectype:4s}", _, get_spectral_class_range(spectype))


O3   44200 (42900.0, 45500.0)
O4   42900 (40900.0, 44200.0)
O5   40900 (39700.0, 42900.0)
O5.5 39700 (39500.0, 40900.0)
O6   39500 (38100.0, 39700.0)
O6.5 38100 (37100.0, 39500.0)
O7   37100 (36100.0, 38100.0)
O7.5 36100 (35100.0, 37100.0)
O8   35100 (34100.0, 36100.0)
O8.5 34100 (33100.0, 35100.0)
O9   33100 (32000.0, 34100.0)
O9.5 32000 (29700.0, 33100.0)
B0   29700 (25400.0, 32000.0)
B1   25400 (22000.0, 29700.0)
B2   22000 (17600.0, 25400.0)
B3   17600 (15400.0, 22000.0)
B5   15400 (14500.0, 17600.0)
B6   14500 (13400.0, 15400.0)
B7   13400 (11400.0, 14500.0)
B8   11400 (10500.0, 13400.0)
B9   10500 (9600.0, 11400.0)
A0   9600 (9330.0, 10500.0)
A1   9330 (9040.0, 9600.0)
A2   9040 (8750.0, 9330.0)
A3   8750 (8480.0, 9040.0)
A4   8480 (8180.0, 8750.0)
A5   8180 (7920.0, 8480.0)
A7   7920 (7220.0, 8180.0)
F0   7220 (7030.0, 7920.0)
F2   7030 (6910.0, 7220.0)
F3   6910 (6640.0, 7030.0)
F5   6640 (6510.0, 6910.0)
F6   6510 (6340.0, 6640.0)
F7   6340 (6160.0, 6510.0)
F8   6160 (5930.0, 

In [ ]:
standardize_spectral_type("s/sdM5"), _parse_spt("dM5V")

('M5', 65.0)

In [ ]:
get_spectral_class_range("dM5V")

(2850.0, 3200.0)

In [ ]:
Counter(e_emc["st_spectype"])

TypeError: unhashable type: 'MaskedConstant'

In [ ]:
l = ""
for key, val in Counter(input_table["st_spectype"]).items():
    print(standardize_spectral_type(key))


K1.5
K5
M3.5
A0
G3
K1
F5
A0
K0
A1
A7
G9
K4
K2
K1
K5
G9.5
K0
B6
K1
K0.5
B8
A4
B7
A1
F2
B8
G0
G0
A1.5
A3
A1
A5
A2.5
A5
B6
F0
A5
A3

A5
A0
F1
F7
B8
G9
A6
F9
F5
G0
F8
F7
A5
G5
G3
F2
K1
F8.5
G4
F6
G2
G2
F9
M2
M1.5
A8
G2
F5
K1
A5
M0.5
G0
K3
F3
G1.5
K0
K2
G3
K5
K2
G3
M3.5
A7
F
A1.5
M1.0
K0
G8
B8
A4
M4
A5
M3
F9
G8
G8
K2
K7
M3.5
G5
K6.5
G8
G2
M7
B9
O
F8
M0
A0
B3
K2.5
K5
K7
M
F3
M4.0
K1
M2.5
K0
K4
M3.5
M2.5
M2
M2
M3
M4.0
M1.5
M4.0
M4
M4.5
M3.0
M2
M4.5
K3
K5
K7
M3.85
M3.14
M2.0
M
M1.5
M4.20
M4.01
M4.1
M1.0
K6
M3.39
M3.74
K7.0
M4.32
M0.5
M2
M
K1
M2
M2.4
M2
M4.0
F0
M5.5
M3.5
M1.0
M2.5
M5
M0.5
M5
M1
M2
G2
A8
A2
A1
A1
G5
K0
M2
G6
F1
K2
K2
K1
A1
A7
G8
A1
G5
F1
K0
F0
K1
G8
K3
K1
K2
G8.5
K1.5
K0
G8
K1.5
G9.5
B8
G8
K3
F5
A5
A2
K2
G9
B9
K0
G8
A0
K0
G7
G9.5
K0
G8
K0
G8
G7
A3
G5
G1
G9.5
K2
G9
A7
K2
G8
K0
F5
G9
F5
F5
K0
A8
K1
G8
A0
A6
G9.5
B8
G9
F8
K0
A1
K1.5
G8.5
G7
G9.5
F2
A2
A3
K1
A7
G9.5
G8
A2
K1
A9
F0
K1
A2
K2
F2
F8.5
F2
F2
K2
K3
A1
A2
F3
A4
A2
A7
F0
G5
B9.5
B8.5
G6
F4
A2
F7
A5
F8
A5
K0
F3
F9.5
F5
F8.5


In [ ]:
sbd = Table.read("./simbad_params.txt", format="ascii")
for i in sbd["sp_type"]:
    print(standardize_spectral_type(i))






F5
F3
K7

G1


K4
G5




M2
M
A8

L3.5
K8




K3
F5
M0
K2

F6







G8
G8





A1
K5
F5




G5
















G0

M4.5



G5




F6



M0






L
M3.5
T8

K0


F7













G4

G0
M2.5







G1
M7.5



M7.25




F6














K0
G7
F5




F0
T5








F5


F7


F2

F






L2

F5

M5.75














L9





K4


K5


M9



G0






F8




K0






M4

F8

K3





F6

F7
G5


K1

K1.5





G8



F6











F8

G5
K0



G1

F2







K2
G3

K0
G8

G0




G5




M1

F2









F4

M3.4

L8.5





F9.5




G2

F6
M8.3


F5


A
G3
M1
G5

G5




M1




Y0

G0

F2
M3













F3











M0

M7
G0

F6
L1.7




G5





K0

M5.5


F5




K5.5
F5
M1














K2





M8.25


F5

G2
K6



F1



K3







K0
G5
M5.25

G1.5



M1


G6
L1

F2

M1


M3

M4.5

K1


M5.0




A1








M0











G2







L5





G9
A2










M4.75


M3.7
K9




G0

F3

L0.1
G8





G0





G3
F5
G8

M1

M8





F1

F8
G7








G8

M1


F5

F9
G1




G8





G5
G3



M6

Y2
K2

M9

In [ ]:
_parse_spt("O4")

4.0

In [ ]:
e_emc.sort("p", reverse=False)
e_emc["exo-mercat_name", "eu_name", "bestmass", "catalog" , "discovery_method","p", "status"][~e_emc["p"].mask & ~(e_emc["catalog"] == "eu")].show_in_notebook()

DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, corner_renderer=None, default_render…

In [ ]:
e_emc.colnames

['exo-mercat_name',
 'nasa_name',
 'toi_name',
 'epic_name',
 'eu_name',
 'oec_name',
 'host',
 'letter',
 'main_id',
 'binary',
 'main_id_ra',
 'main_id_dec',
 'mass',
 'mass_max',
 'mass_min',
 'mass_url',
 'msini',
 'msini_max',
 'msini_min',
 'msini_url',
 'bestmass',
 'bestmass_max',
 'bestmass_min',
 'bestmass_url',
 'bestmass_provenance',
 'p',
 'p_max',
 'p_min',
 'p_url',
 'r',
 'r_max',
 'r_min',
 'r_url',
 'a',
 'a_max',
 'a_min',
 'a_url',
 'e',
 'e_max',
 'e_min',
 'e_url',
 'i',
 'i_max',
 'i_min',
 'i_url',
 'discovery_method',
 'status',
 'checked_status_string',
 'original_status_string',
 'confirmed',
 'discovery_year',
 'main_id_aliases',
 'main_id_provenance',
 'angular_separation_flag',
 'angular_separation',
 'catalog',
 'duplicate_catalog_flag',
 'duplicate_names',
 'binary_coordinate_mismatch_flag',
 'binary_complex_system_flag',
 'coordinate_mismatch_flag',
 'coordinate_mismatch',
 'period_mismatch_flag',
 'fallback_merge_flag',
 'misnamed_duplicates_flag',
 'r